# Taller 1: Análisis de Clustering de Sismicidad en Colombia
**Autor:** Ailyn Sofía
**Metodología:** CRISP-DM

## Introducción
Colombia es un país en constante movimiento. Ubicado en una zona de alta complejidad tectónica (Placas de Nazca, Suramericana y Caribe), el país registra miles de eventos sísmicos. Este notebook documenta el proceso de identificación de patrones sísmicos mediante **Machine Learning (K-Means Clustering)**.

## 1. Data Understanding (Entendimiento de Datos)
Cargamos las librerías necesarias y el dataset oficial del Servicio Geológico Colombiano.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

plt.style.use('ggplot')
sns.set_palette('viridis')

# Cargar datos
df = pd.read_csv('../data/raw/earthquakes_colombia.csv')
print(f'Registros cargados: {len(df)}')
df.head()

### Análisis Exploratorio (EDA)
Buscamos heterogeneidad en la profundidad y magnitud, lo que justifica la segmentación por clusters.

In [ ]:
critical_vars = ['latitude', 'longitude', 'depth', 'mag']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['depth'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribución de Profundidad (km)')

sns.histplot(df['mag'], kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribución de Magnitud')

plt.show()

## 2. Data Preparation (Preparación de Datos)
Dado que la profundidad se mide en km (0-600) y las coordenadas en grados (-80 a 15), es crítico realizar un **Escalamiento Estándar** para que el algoritmo no ignore la profundidad.

In [ ]:
features = ['latitude', 'longitude', 'depth']
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[features])

print('Muestra de datos escalados (Normalizados):')
print(df_scaled[:5])

## 3. Modeling (Modelado)
Utilizamos el método del Codo (Inercia) y el Score de Silueta para determinar el número óptimo de clusters (K).

In [ ]:
inertia = []
silhouette_avg = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(df_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_avg.append(silhouette_score(df_scaled, cluster_labels))

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(K_range, inertia, 'bx-')
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inercia (Codo)', color='b')

ax2 = ax1.twinx()
ax2.plot(K_range, silhouette_avg, 'ro-')
ax2.set_ylabel('Silhouette Score', color='r')

plt.title('Selección de K: Codo vs Silhouette')
plt.show()

### Modelo Final: K=5
Seleccionamos K=5 para capturar la diversidad regional (Norte, Sur, Pacífico, Centro y Nido de Bucaramanga).

In [ ]:
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(df_scaled)

print('Distribución de eventos por cluster:')
print(df['cluster'].value_counts())

## 4. Evaluation (Evaluación)
Visualizamos geográficamente los clusters identificados.

In [ ]:
plt.figure(figsize=(10, 12))
sns.scatterplot(data=df, x='longitude', y='latitude', hue='cluster', palette='viridis', alpha=0.6, s=20)
plt.title('Mapa de Clusters Sísmicos en Colombia (Vista 2D)')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 5. Análisis Geográfico-Social
De acuerdo a los resultados, identificamos las siguientes zonas críticas:

1. **Nido sísmico de Bucaramanga (Santander):** Zona de alta profundidad, única en el mundo por su concentración.
2. **Costa Pacífica (Chocó/Nariño):** Sismos superficiales con alto potencial destructivo en zonas socialmente vulnerables.
3. **Eje Andino:** Concentración poblacional sobre fallas activas.
4. **Frontera Sur y Caribe:** Zonas de acomodación cortical.

Para una experiencia interactiva completa, visite el Dashboard desplegado:
**[https://colombia-seismic-clusters.vercel.app](https://colombia-seismic-clusters.vercel.app)**